# N121 · Aho–Corasick多模式匹配

**目标：** 通过失败链接共享多个模式的扫描工作。

**先修：** N077, N078, N081, N118。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Trie；BFS建fail；输出传播；重叠匹配；流式状态；输出量复杂度。

## 从问题到算法

多模式搜索不能为每个词重新扫描整个文本。Trie把公共前缀合并；失败指针把当前文本后缀转移到仍可能匹配的最长模式前缀。一个状态可能同时结束多个模式，例如she也以he结尾，所以输出必须包含失败链上的终止模式。

## 最小实现

**本章接口：** `AhoCorasick.build(patterns)`、`AhoCorasick.scan(text)`、`StreamChecker.query(letter)`

In [1]:
from collections import deque

class AhoCorasick:
    def __init__(self,patterns=None):
        self.build([] if patterns is None else patterns)
    def build(self,patterns):
        self.patterns=list(patterns); self.next=[{}]; self.fail=[0]; self.output=[[]]
        for pid,pattern in enumerate(self.patterns):
            if not pattern: raise ValueError('empty patterns are excluded')
            node=0
            for c in pattern:
                if c not in self.next[node]:
                    self.next[node][c]=len(self.next); self.next.append({}); self.fail.append(0); self.output.append([])
                node=self.next[node][c]
            self.output[node].append(pid)
        queue=deque(self.next[0].values())
        while queue:
            u=queue.popleft()
            for c,v in self.next[u].items():
                f=self.fail[u]
                while f and c not in self.next[f]: f=self.fail[f]
                self.fail[v]=self.next[f].get(c,0)
                self.output[v].extend(self.output[self.fail[v]]); queue.append(v)
        return self
    def step(self,state,letter):
        while state and letter not in self.next[state]: state=self.fail[state]
        return self.next[state].get(letter,0)
    def scan(self,text):
        state=0; matches=[]
        for i,c in enumerate(text):
            state=self.step(state,c)
            matches.extend((i,pid) for pid in self.output[state])
        return matches

class StreamChecker:
    def __init__(self,words): self.automaton=AhoCorasick(words); self.state=0
    def query(self,letter):
        if len(letter)!=1: raise ValueError('one character per query')
        self.state=self.automaton.step(self.state,letter)
        return bool(self.automaton.output[self.state])

## 正确性、前提与复杂度

状态表示已经读取文本的最长后缀，且该后缀是某模式前缀。失败链依次枚举更短候选，找到转移即恢复不变量。沿失败链继承输出保留后缀匹配和重叠匹配；重复模式保留不同ID，不能无意去重。

**代价：** 扫描O(T+匹配输出数)平均字典操作；单次query最坏可沿长失败链，总流式回退可摊还。这个稀疏实现构建保守O(M·L+输出列表复制量)，M为Trie节点数、L最长模式；复制后缀输出可能增大空间，拓展可用输出链接避免复制。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

patterns=['he','she','hers','his']; automaton=AhoCorasick(patterns)
show_table(['状态','Trie出边','失败指针','模式ID'],[(i,automaton.next[i],automaton.fail[i],automaton.output[i]) for i in range(len(automaton.next))])
print('ushers的(结束下标,模式ID)',automaton.scan('ushers'))

状态,Trie出边,失败指针,模式ID
0,"{'h': 1, 's': 3}",0,[]
1,"{'e': 2, 'i': 8}",0,[]
2,{'r': 6},0,[0]
3,{'h': 4},0,[]
4,{'e': 5},1,[]
5,{},2,"[1, 0]"
6,{'s': 7},0,[]
7,{},3,[2]
8,{'s': 9},0,[]
9,{},3,[3]


ushers的(结束下标,模式ID) [(3, 1), (3, 0), (5, 2)]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
patterns=['he','she','hers','his']; ac=AhoCorasick(patterns)
assert sorted(ac.scan('ushers'))==[(3,0),(3,1),(5,2)]
ac=AhoCorasick(['a','a','ba'])
assert sorted(ac.scan('ba'))==[(1,0),(1,1),(1,2)]
from itertools import product
patterns=['a','ab','bab','b','aba']; ac=AhoCorasick(patterns)
for n in range(8):
    for chars in product('ab',repeat=n):
        text=''.join(chars)
        expected=[(i,pid) for i in range(n) for pid,p in enumerate(patterns) if text[:i+1].endswith(p)]
        assert sorted(ac.scan(text))==sorted(expected)
        stream=StreamChecker(patterns)
        assert [stream.query(c) for c in text]==[any(text[:i+1].endswith(p) for p in patterns) for i in range(n)]
print('N121: 所有本章断言通过')

N121: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 与倒序Trie解1032对照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明总复杂度必须计入报告匹配数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1032. Stream of Characters（extension）：[官方入口](https://leetcode.com/problems/stream-of-characters/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。